# Storm maps: dynamic run, linear sum and buoys

The largest storms of the dynamic run (08) seen three ways, on the hours whose case kept
its map (`save_map`, a random ~5 % of the hours):

- the **dynamic SnapWave map**, with the buoys coloured by the observed Hs on the same scale;
- the **linear sum** (metamodel reconstruction, spectral mode) at the sites of one set, and
  its difference to the dynamic run at the same sites;
- the **time series at the buoys** through each storm: buoy, dynamic run, linear sum bulk
  and spectral (`tm01` only for the buoy and the sums: it is a stat of the summed spectrum).

Storms are the peaks of the goal-mean offshore Hs, at least `min_gap` apart, kept when a
saved map lies within `max_lag` of the peak. SnapWave clips the boundary spread at 55°
(notebook 08), the reconstruction does not: in broad-spread hours the buoys are the referee.

**Prerequisites:** dynamic run postprocessed (08, `Dynamic_Cases_<period>.nc`), fitted
models (06), buoy files (03). The reconstructions are read from the cache
(`tmp/reconstruct/`, filled by 09 or a script); anything missing is computed here.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr
from utils import metamodel as mm
from utils import simulation as sim
from utils import validation as val
from utils import viz
from utils.config import get_offshore_wave, open_offshore_waves

## Configuration

In [ ]:
period = "2018"  # a year of the dynamic run
offshore_name = "WHACS_cal"
wind = False  # dynamic run with or without wind (08); the metamodel has none
mda_n = 800  # must match a fitted mda_candidates entry
map_set = "isobaths"  # metamodel set drawn on the maps
buoy_set = "buoys"  # metamodel set at the buoys
wl = "forcing"  # same water level as the dynamic run
n_storms = 4
min_gap = "4D"  # least time between two storm peaks
max_lag = "6h"  # saved map at most this far from the peak
window = pd.Timedelta("3D")  # half-width of the buoy time series
extent = None  # (lon_min, lon_max, lat_min, lat_max) to zoom the maps; None = the mesh
series_variables = ["hs", "tp", "tm01", "dir"]

cases = sim.cases_dir("dynamic", offshore=offshore_name, wind=wind)
dynamic_path = cases / f"Dynamic_Cases_{period}.nc"
print(f"dynamic run : {dynamic_path}")

## 1 · Storms with a saved map

In [ ]:
offshore = open_offshore_waves(entry=get_offshore_wave(offshore_name)).sel(time=period).load()
map_hours = sim.saved_map_hours(cases)
storms = val.storm_peaks(offshore, n_storms, min_gap=min_gap, map_hours=map_hours, max_lag=max_lag)
print(f"{len(map_hours)} hours with a saved map")
storms.round(2)

## 2 · Dynamic run, reconstructions and buoys

In [ ]:
# Resources for the reconstructions (at most 75% of this machine). Edit res if wanted.
res = mm.reconstruction_resources(
    period, mda_n, metamodel=map_set, spectral=True,
    offshore_name=offshore_name, save_spectra=False, wl=wl,
)

In [ ]:
dynamic = xr.open_dataset(dynamic_path).sel(time=period).load()
recon = {
    (mm_set, mode): mm.reconstruct(
        period, mda_n, metamodel=mm_set, spectral=spectral,
        offshore_name=offshore_name, resources=res, save_spectra=False, wl=wl,
    )
    for mm_set in (map_set, buoy_set)
    for mode, spectral in {"bulk": False, "spectral": True}.items()
}
buoy = val.load_buoy_observations(period, variables=series_variables)
print(f"dynamic {dynamic.sizes['sites']} sites; buoys with data: {list(buoy.sites.values)}")

## 3 · Maps

Left: the dynamic run (buoys: observed Hs, same colour scale). Middle: the spectral linear
sum at the sites of `map_set`. Right: linear sum minus dynamic (red: the sum is higher).

In [ ]:
for storm in storms.itertuples():
    viz.plot_storm_hour(
        cases / f"{storm.map_hour:%Y%m%d %H%M%S}", dynamic, recon[map_set, "spectral"], buoy,
        estimate_label="Linear sum (spectral)", extent=extent,
        title=f"Storm peak {storm.peak:%Y-%m-%d %H:%M} (map {storm.map_hour:%H:%M}), "
              f"goal-mean offshore Hs {storm.goal_mean_hs:.1f} m",
    )
    plt.show()

## 4 · The buoys through each storm

Buoy, dynamic run and the linear sums (bulk, spectral) from `window` before to `window`
after each peak, at the buoys with data then; the table gives the Hs peak of each and its lag.

In [ ]:
for storm in storms.itertuples():
    span = slice(storm.peak - window, storm.peak + window)
    obs = buoy.sel(time=span)
    sites = [s for s in obs.sites.values if int(obs["hs"].sel(sites=s).notnull().sum())]
    if not sites:
        print(f"{storm.peak:%Y-%m-%d}: no buoy data")
        continue
    estimates = {
        val.DYNAMIC_REFERENCE_LABEL: dynamic.sel(time=span, sites=sites),
        "Linear sum (bulk)": recon[buoy_set, "bulk"].sel(time=span, sites=sites),
        "Linear sum (spectral)": recon[buoy_set, "spectral"].sel(time=span, sites=sites),
    }
    display(val.event_peaks(obs.sel(sites=sites), estimates).round(2))
    for site in sites:
        val.plot_buoy_timeseries(obs, estimates, site, variables=series_variables)
        plt.suptitle(f"{site} - storm {storm.peak:%Y-%m-%d}")
        plt.show()